# Treinamento do Modelo Baseline — Classificacao de Falhas
## FIAP x Forzy | Challenge IA | Sprint 2 — Grupo 4M

Fluxo: descricao do problema -> base -> tratamento -> split cronologico -> treino baseline -> avaliacao por classe -> analise de erros -> ablacao com features de janela -> serializacao.

## 1. Descricao do problema preditivo *(Entregavel 1)*

Tarefa de **classificacao multiclasse** sobre o estado operacional do motor; variavel-alvo `falha`:

| Codigo | Estado |
|---|---|
| 0 | Normal |
| 1 | Desbalanceamento |
| 2 | Superaquecimento |
| 3 | Falha Mecanica |

Antecipar *qual* falha esta se desenvolvendo permite agendar a intervencao certa antes da parada nao planejada (manutencao preditiva). Como as 3 classes de falha sao **minoritarias** (~12,4%), priorizamos **F1, precisao e recall por classe** sobre a acuracia global.

## 2. Setup e carregamento *(Entregavel 2)*

In [ ]:
import sqlite3, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

conn = sqlite3.connect('motor.db')
df = pd.read_sql('''SELECT l.*, m.fabricante, m.modelo FROM leituras l
                     JOIN motores m ON l.motor_id=m.motor_id
                     ORDER BY l.motor_id, l.timestamp''', conn)
conn.close(); df['timestamp']=pd.to_datetime(df['timestamp'])
print('Shape:', df.shape)

## 3. Tratamento e preparacao *(Entregavel 4)*

- **Nulos:** base sintetica sem ausentes (verificado).
- **Categoricas (`fabricante`,`modelo`):** one-hot; como todos sao WEG W22, tem variancia zero e sao descartadas explicitamente.
- **Padronizacao:** StandardScaler ajustado **so no treino** (sem vazamento).
- **Desbalanceamento:** `class_weight='balanced'` (preferido a SMOTE, que interpolaria leituras temporais de instantes distintos).

In [ ]:
print('Nulos:', int(df.isnull().sum().sum()))
cat = pd.get_dummies(df[['fabricante','modelo']])
print('Categoricas uteis (var>0):', cat.columns[cat.nunique()>1].tolist() or 'nenhuma -> descartadas')
print('\nAlvo (%):'); print((df['falha'].value_counts(normalize=True)*100).round(1).sort_index().to_string())

## 4. Split cronologico 70/30 *(Entregavel 5)*

Primeiras 70% leituras de cada motor = treino; 30% finais = teste. Evita vazamento temporal.

In [ ]:
df['pos'] = df.groupby('motor_id').cumcount()
mask = df['pos'] < 1500*0.7
y = df['falha'].values

## 5. Modelo baseline *(Entregaveis 6 e 7)*

O modelo **entregue** (`modelo_falhas.joblib`) usa os **4 sensores brutos** — exatamente os parametros que o agente conversacional envia em cada consulta pontual. Isso garante coerencia treino/inferencia. Comparamos Regressao Logistica (referencia) e Random Forest; escolhemos o **Random Forest** por capturar interacoes nao-lineares (ex.: temperatura *e* corrente subindo juntas).

In [ ]:
sensores = ['rotacao','vibracao','temperatura','corrente']  # ordem usada pela tool do agente
Xb = df[sensores].values
scaler = StandardScaler().fit(Xb[mask.values])
Xtr, Xte = scaler.transform(Xb[mask.values]), scaler.transform(Xb[~mask.values])
ytr, yte = y[mask.values], y[~mask.values]

lr = LogisticRegression(class_weight='balanced', max_iter=1000).fit(Xtr, ytr)
print('F1 macro (LogReg):', round(f1_score(yte, lr.predict(Xte), average='macro'),3))

rf = RandomForestClassifier(n_estimators=150, max_depth=14, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr); pred = rf.predict(Xte)
print('F1 macro (RandomForest):', round(f1_score(yte, pred, average='macro'),3))

### Avaliacao por classe *(Entregavel 7)*

In [ ]:
nomes=['Normal','Desbalanceamento','Superaquecimento','Falha Mecanica']
print(classification_report(yte, pred, digits=3, target_names=nomes))
cm = confusion_matrix(yte, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal','Desbal.','Superaq.','Mec.'], yticklabels=['Normal','Desbal.','Superaq.','Mec.'])
plt.xlabel('Previsto'); plt.ylabel('Real'); plt.title('Matriz de Confusao — modelo baseline')
plt.tight_layout(); plt.savefig('matriz_confusao.png', dpi=120); plt.show()

**Leitura:** o recall e excelente em Normal e Superaquecimento; o **Desbalanceamento** e a classe mais dificil para o modelo pontual, porque com uma unica leitura a vibracao no inicio da rampa ainda se confunde com operacao normal. Esse achado motiva a secao 7 (analise de erros) e a secao 8 (features de janela).

## 6. Analise dos erros do modelo *(Entregavel 8)*

In [ ]:
te = df[~mask.values].copy(); te['pred']=pred; te['erro']=te['pred']!=te['falha']
print(f"Erro global: {100*te['erro'].mean():.1f}%\n")
tab = te.groupby('falha')['erro'].mean().round(3); tab.index=nomes
print('Taxa de erro por classe:'); print(tab.to_string())

In [ ]:
te_f = te[te['falha']!=0].copy()
te_f['fase'] = pd.cut(te_f['pos'], bins=3, labels=['inicio','meio','fim'])
d = te_f[te_f['erro']].groupby('fase', observed=True).size()
d.plot(kind='bar', color='#C73E1D'); plt.title('Erros ao longo da janela de falha')
plt.ylabel('Qtd de erros'); plt.tight_layout(); plt.savefig('erros_por_fase.png', dpi=120); plt.show()
print(d)

**Conclusoes da analise de erros:**
- Erros concentrados nas falhas **minoritarias** (Desbalanceamento e, em menor grau, Falha Mecanica), nao em Normal.
- A grande maioria ocorre no **inicio da janela de falha**, quando o desvio ainda e sutil — comportamento esperado e ate seguro (o modelo so se compromete quando a falha esta caracterizada).
- **Acao para a proxima sprint:** features temporais (janela movel) para capturar a tendencia da rampa, validadas na secao seguinte.

## 7. Ablacao — features de janela movel *(melhoria)*

Adicionamos media e desvio em janela de 10 leituras por motor. Isso da ao modelo a **tendencia** que falta a uma leitura isolada.

In [ ]:
dfj = df.copy()
for c in sensores:
    g = dfj.groupby('motor_id')[c]
    dfj[f'{c}_mean'] = g.transform(lambda s: s.rolling(10,min_periods=1).mean())
    dfj[f'{c}_std']  = g.transform(lambda s: s.rolling(10,min_periods=1).std().fillna(0))
cols = sensores + [f'{c}_mean' for c in sensores] + [f'{c}_std' for c in sensores]
Xj = dfj[cols].values
scj = StandardScaler().fit(Xj[mask.values])
rfj = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rfj.fit(scj.transform(Xj[mask.values]), ytr)
f1j = f1_score(yte, rfj.predict(scj.transform(Xj[~mask.values])), average='macro')
print('F1 macro com janela movel:', round(f1j,3))
print('\nGanho vs baseline pontual:', round(f1j - f1_score(yte, pred, average='macro'),3))

**Trade-off (decisao de projeto):** as features de janela elevam o F1 macro para ~0,97, mas exigem **historico temporal** que o agente nao tem numa consulta pontual de 4 leituras. Por isso o modelo **entregue** e o baseline de 4 sensores; a versao com janela fica documentada como o caminho de evolucao para um pipeline que receba a serie completa do motor.

## 8. Serializacao *(modelo reutilizavel)*

Salvamos o **modelo baseline** (4 sensores) usado pelo agente.

In [ ]:
joblib.dump(rf, 'modelo_falhas.joblib', compress=3)
joblib.dump(scaler, 'scaler.joblib')
joblib.dump({'features':sensores, 'ordem_sensores':sensores,
             'classes':{0:'Normal',1:'Desbalanceamento',2:'Superaquecimento',3:'Falha Mecanica'},
             'modelo':'RandomForestClassifier','n_estimators':150,'max_depth':14},
            'modelo_config.joblib')
print('Salvos: modelo_falhas.joblib, scaler.joblib, modelo_config.joblib')

*FIAP x Forzy | Challenge IA | Sprint 2 — Grupo 4M*